In [ ]:
!pip install --upgrade binance-historical-data plotly -q
!pip install requests

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from binance_historical_data import BinanceDataDumper
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
warnings.filterwarnings('ignore')

In [ ]:
def get_list_all_trading_pairs():
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_list_all_trading_pairs()

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h"):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency= data_frequency,
    )
    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()
    data_dumper.dump_data(
        tickers = ticker,
        date_start = date_start,
        date_end = date_end,
        is_to_update_existing = False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency = "1h"):
    daily_path = os.path.join(os.getcwd(), f"spot/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"spot/monthly/klines/{ticker}/{data_frequency}")
    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)
    all_files = daily_files + monthly_files
    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker}")
        return None
    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)
    return data


In [ ]:
data_frequency = "1m"
start_date = "2024-01-01"
end_date = "2025-07-01"
tickers =['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'XRPUSDT', 'TRXUSDT', 'ADAUSDT', 'DOTUSDT', 'LTCUSDT']




In [ ]:
download_ticker(tickers[0], start_date, end_date, data_frequency)
crypto_data = process_csv_files(tickers[0], data_frequency)


---> Found overall tickers: 594
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1m
---> Start Date: 20240101
---> End Date: 20250701


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 0 months 0 days
---> For BTCUSD new data saved for: 0 months 0 days


In [ ]:
data = crypto_data[['Open','Close','volume']].reset_index(drop=True)
data.columns = ['Open','Close', 'Volume']
data

,Open,Close,Volume
0,42298.62,42320.00,21.16779
1,42319.99,42325.50,21.60391
2,42325.50,42367.99,30.50730
3,42368.00,42397.23,46.05107
4,42397.22,42409.20,32.26766
...,...,...,...
789096,105594.33,105664.92,5.33736
789097,105664.93,105664.92,2.04199
789098,105664.92,105681.14,2.67159
789099,105681.14,105681.14,6.27463


In [ ]:
import pandas as pd
import numpy as np

def backtest_volume_strategy(df, lookback=5, volume_multiplier=2, fee_percent=0.1):
    """
    Backtest a strategy that buys/sells based on the trend (Open - Close) of the nearest high-volume bar, including trading fees.
    Parameters:
        df: DataFrame with OHLCV data (columns: Open, High, Low, Close, Volume)
        lookback: Periods for calculating average volume
        volume_multiplier: Multiplier to determine high-volume bars
        fee_percent: Transaction fee as a percentage (e.g., 0.1 for 0.1%)
    Returns:
        DataFrame with trade signals, fees, and performance metrics
    """
    # Ensure DataFrame has required columns
    required_cols = ['Open', 'Close', 'Volume']
    if not all(col in df.columns for col in required_cols):
        raise ValueError("DataFrame must contain OHLCV columns")

    # Convert fee percentage to decimal
    fee = fee_percent / 100

    # Calculate average volume
    df['Avg_Volume'] = df['Volume'].rolling(window=lookback).mean()

    # Identify high-volume bars (>= 2 * average volume)
    df['High_Volume'] = df['Volume'] >= volume_multiplier * df['Avg_Volume']

    # Initialize columns
    df['Signal'] = 0  # 1 for buy, -1 for sell, 0 for hold
    df['Position'] = 0
    df['Returns'] = df['Close'].pct_change()
    df['Strategy_Returns'] = 0.0
    df['Trade_Fee'] = 0.0

    # Generate signals based on the trend of the most recent high-volume bar
    for i in range(1, len(df)):
        if pd.isna(df['Avg_Volume'].iloc[i]):
            continue  # Skip if not enough data for average volume

        # Look for the most recent high-volume bar up to the previous row
        recent_high_vol = df.iloc[:i][df['High_Volume'].iloc[:i]].index
        if not recent_high_vol.empty:
            last_high_vol_idx = recent_high_vol[-1]
            # Determine trend of the high-volume bar (Open - Close)
            high_vol_open = df['Open'].iloc[last_high_vol_idx]
            high_vol_close = df['Close'].iloc[last_high_vol_idx]

            # Generate signal based on the trend of the high-volume bar
            if high_vol_close > high_vol_open:
                df.at[df.index[i], 'Signal'] = -1  # Buy (bullish trend)
            elif high_vol_close < high_vol_open:
                df.at[df.index[i], 'Signal'] = 1  # Sell (bearish trend)

    # Shift signals to apply them to the next row
    df['Position'] = df['Signal'].shift(1).fillna(0)

    # Calculate fees and strategy returns
    for i in range(1, len(df)):
        # Apply fees when position changes
        prev_position = df['Position'].iloc[i-1]
        curr_position = df['Position'].iloc[i]
        if curr_position != prev_position:
            df.at[df.index[i], 'Trade_Fee'] = fee

        # Calculate strategy returns for the current row using the position from the previous row
        if not pd.isna(df['Returns'].iloc[i]) and curr_position != 0:
            df.at[df.index[i], 'Strategy_Returns'] = curr_position * df['Returns'].iloc[i] - df['Trade_Fee'].iloc[i]
        else:
            df.at[df.index[i], 'Strategy_Returns'] = -df['Trade_Fee'].iloc[i]

    # Calculate cumulative returns
    df['Cumulative_Market_Returns'] = (1 + df['Returns']).cumprod() - 1
    df['Cumulative_Strategy_Returns'] = (1 + df['Strategy_Returns']).cumprod() - 1

    # Calculate performance metrics
    total_trades = (df['Signal'] != 0).sum()
    total_fees = df['Trade_Fee'].sum()
    win_rate = (df['Strategy_Returns'] > 0).sum() / total_trades if total_trades > 0 else 0
    sharpe_ratio = (df['Strategy_Returns'].mean() / df['Strategy_Returns'].std()) * np.sqrt(252) if df['Strategy_Returns'].std() != 0 else 0

    # Print summary
    print(f"Total Trades: {total_trades}")
    print(f"Total Fees Paid: {total_fees:.4f}")
    print(f"Win Rate: {win_rate:.2%}")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Cumulative Strategy Return: {df['Cumulative_Strategy_Returns'].iloc[-1]:.2%}")

    return df

In [ ]:
backtest_volume_strategy(data).to_csv('result.csv')

Total Trades: 786357
Total Fees Paid: 23.4030
Win Rate: 47.91%
Sharpe Ratio: -0.57
Cumulative Strategy Return: -100.00%
